# RNN and bidirectional RNN

Run the cells in order.

Use the course-provided CoNLL files in `data/conll2003/`; see the README for setup.

## CoNLL sentences

We will compare a forward RNN with a bidirectional RNN on the same 1,000 training and 200 validation sentences.

This experiment learns 32-dimensional word embeddings from the training data. Both models start with the same embedding weights.

In [ ]:
from pathlib import Path
import random, copy, time, json
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from seqeval.metrics import precision_score, recall_score, f1_score
from seqeval.scheme import IOB2

torch.set_num_threads(1)
SEED = 17
LABELS = "O B-PER I-PER B-ORG I-ORG B-LOC I-LOC B-MISC I-MISC".split()
DATA = Path("data/conll2003")
def read_conll(path):
    examples, words, tags = [], [], []
    for line in path.read_text().splitlines() + [""]:
        fields = line.split()
        if fields and fields[0] == "-DOCSTART-":
            continue
        if fields:
            words.append(fields[0]); tags.append(fields[-1])
        elif words:
            examples.append((words, tags)); words, tags = [], []
    return examples
rng = random.Random(SEED)
train = rng.sample(read_conll(DATA / "train.txt"), 1000)
valid = rng.sample(read_conll(DATA / "valid.txt"), 200)
vocab = {"<PAD>": 0, "<UNK>": 1}
for words, _ in train:
    for word in words:
        vocab.setdefault(word.lower(), len(vocab))
def batch(examples):
    ids = [torch.tensor([vocab.get(w.lower(), 1) for w in words]) for words, _ in examples]
    tags = [torch.tensor([LABELS.index(t) for t in gold]) for _, gold in examples]
    lengths = torch.tensor([len(x) for x in ids])
    return (pad_sequence(ids, batch_first=True, padding_value=0),
            pad_sequence(tags, batch_first=True, padding_value=-100), lengths)
torch.manual_seed(SEED)
initial_embeddings = torch.randn(len(vocab), 32) * 0.1
initial_embeddings[0].zero_()
print("Train:", len(train), "validation:", len(valid), "vocabulary:", len(vocab))
print(list(zip(*train[0])))

**Question:** Read an entity in the printed sentence. Which words before and after it could help predict its type?

## Forward and bidirectional models

A forward RNN uses the current word and its left context. A bidirectional RNN combines a forward state with a backward state at every word.

Each direction has 32 hidden features. The bidirectional classifier receives 64 features.

In [ ]:
class Tagger(nn.Module):
    def __init__(self, bidirectional=False):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            initial_embeddings.clone(), freeze=False, padding_idx=0)
        self.rnn = nn.RNN(32, 32, batch_first=True, bidirectional=bidirectional)
        self.output = nn.Linear(64 if bidirectional else 32, len(LABELS))

    def forward(self, ids, lengths):
        vectors = self.embedding(ids)
        packed = pack_padded_sequence(vectors, lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        packed_states, _ = self.rnn(packed)
        states, _ = pad_packed_sequence(packed_states, batch_first=True,
                                        total_length=ids.shape[1])
        return self.output(states)

torch.manual_seed(SEED)
forward_model = Tagger(False)
torch.manual_seed(SEED)
bidirectional_model = Tagger(True)
ids, targets, lengths = batch(train[:4])
for model in [forward_model, bidirectional_model]:
    print("Scores:", model(ids, lengths).shape,
          "parameters:", sum(p.numel() for p in model.parameters()))
assert torch.equal(forward_model.embedding.weight, bidirectional_model.embedding.weight)

**Question:** Why does the bidirectional output layer need 64 input features?

## Padding and the backward pass

We pack the sequences so each backward pass starts at the last real word. Padding values stay outside the recurrence.

The loss separately ignores targets marked −100.

In [ ]:
for model in [forward_model, bidirectional_model]:
    model.eval()
    ids, _, lengths = batch(train[:3])
    longer_ids = torch.cat([ids, torch.zeros(len(ids), 5, dtype=torch.long)], dim=1)
    with torch.no_grad():
        original = model(ids, lengths)
        padded = model(longer_ids, lengths)[:, :ids.shape[1]]
    assert torch.allclose(original, padded, atol=1e-6)
print("Appending PAD leaves the real-word scores unchanged in both models.")

**Question:** Keep the true lengths and append more PAD positions. Check the scores again.

## Train both models

Both models use the same batches, six epochs, cross-entropy loss and Adam learning rate. We record the final validation result for one seed.

The bidirectional model also has more parameters. This comparison changes both available context and model capacity.

In [ ]:
def evaluate(model, examples):
    gold, predicted = [], []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(examples), 32):
            subset = examples[start:start+32]
            ids, _, lengths = batch(subset)
            prediction = model(ids, lengths).argmax(-1)
            for row, (words, tags) in zip(prediction, subset):
                gold.append(tags)
                predicted.append([LABELS[i] for i in row[:len(words)].tolist()])
    options = dict(mode="strict", scheme=IOB2, zero_division=0)
    return {"precision": float(precision_score(gold, predicted, **options)),
            "recall": float(recall_score(gold, predicted, **options)),
            "f1": float(f1_score(gold, predicted, **options))}, predicted

orders = []
for epoch in range(6):
    order = list(range(len(train)))
    random.Random(SEED + epoch).shuffle(order)
    orders.append(order)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
real_results = {}
for name, model in [("RNN", forward_model), ("BiRNN", bidirectional_model)]:
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    start_time = time.perf_counter()
    losses = []
    for epoch, order in enumerate(orders):
        model.train()
        total, count = 0.0, 0
        for start in range(0, len(order), 32):
            examples = [train[i] for i in order[start:start+32]]
            ids, targets, lengths = batch(examples)
            optimizer.zero_grad()
            scores = model(ids, lengths)
            loss = criterion(scores.reshape(-1, len(LABELS)), targets.reshape(-1))
            loss.backward()
            optimizer.step()
            tokens = targets.ne(-100).sum().item()
            total += loss.item() * tokens; count += tokens
        losses.append(total / count)
        print(name, "epoch", epoch+1, "loss", round(losses[-1], 3))
    metrics, predicted = evaluate(model, valid)
    real_results[name] = {"metrics": metrics, "predicted": predicted,
                          "parameters": sum(p.numel() for p in model.parameters()),
                          "loss": losses, "seconds": time.perf_counter()-start_time}
    print(name, {k: round(v, 3) for k,v in metrics.items()})
print('RNN_REAL_RESULT:' + json.dumps({name: {k: v for k,v in result.items() if k != 'predicted'} for name,result in real_results.items()}))

**Question:** Run fresh models with another seed, keeping the sampled data fixed. Does the difference persist?

## Validation predictions

Compare entity precision, recall and F1, then inspect sentences where the models disagree.

An exact entity match requires the correct type and both boundaries. Strict IOB2 scoring also checks the tag sequence.

In [ ]:
for name, result in real_results.items():
    print(name, result["metrics"], "parameters:", result["parameters"])
disagreements = [i for i in range(len(valid))
                 if real_results["RNN"]["predicted"][i] != real_results["BiRNN"]["predicted"][i]]
for i in disagreements[:3]:
    words, gold = valid[i]
    print("\n", " ".join(words))
    for word, g, a, b in zip(words, gold, real_results["RNN"]["predicted"][i],
                            real_results["BiRNN"]["predicted"][i]):
        if g != "O" or a != "O" or b != "O":
            print(word, "gold:", g, "RNN:", a, "BiRNN:", b)
view = []
for i in disagreements[:8]:
    words, gold = valid[i]
    view.append({"words": words, "gold": gold,
                 "RNN": real_results["RNN"]["predicted"][i],
                 "BiRNN": real_results["BiRNN"]["predicted"][i]})
print("RNN_REAL_ERRORS:" + json.dumps(view))

**Question:** Find an entity the bidirectional model gets right and the forward model misses. Identify a useful word to its right.